In [1]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 21.4 MB/s eta 0:00:00


In [2]:
import ccxt
from datetime import datetime

exchange = ccxt.coinbase({
    'rateLimit': 1000,
    'enableRateLimit': True,
})

symbol = 'BTC/USD'

# 拉订单簿（前20档，看得更全）
ob = exchange.fetch_order_book(symbol, limit=20)

mid = (ob['bids'][0][0] + ob['asks'][0][0]) / 2

# 算深度（前5档、前10档、前20档）
def calc_imbalance(bids, asks, levels):
    bid_vol = sum([v for p, v in bids[:levels]])
    ask_vol = sum([v for p, v in asks[:levels]])
    imbalance = (bid_vol - ask_vol) / (bid_vol + ask_vol) * 100
    return bid_vol, ask_vol, imbalance

bid5, ask5, imb5 = calc_imbalance(ob['bids'], ob['asks'], 5)
bid10, ask10, imb10 = calc_imbalance(ob['bids'], ob['asks'], 10)
bid20, ask20, imb20 = calc_imbalance(ob['bids'], ob['asks'], 20)

print('=' * 55)
print(f'  {symbol} | {datetime.now().strftime("%H:%M:%S")}')
print(f'  中间价: ${mid:,.2f}')
print('=' * 55)
print(f'  {"档位":<8} {"买方深度":>10} {"卖方深度":>10} {"不平衡度":>10} {"信号":>6}')
print(f'  {"-"*44}')
print(f'  前5档    {bid5:>8.4f}   {ask5:>8.4f}   {imb5:>+8.2f}%   ', end='')
print('🟢' if imb5 > 10 else ('🔴' if imb5 < -10 else '⚪'))
print(f'  前10档   {bid10:>8.4f}   {ask10:>8.4f}   {imb10:>+8.2f}%   ', end='')
print('🟢' if imb10 > 10 else ('🔴' if imb10 < -10 else '⚪'))
print(f'  前20档   {bid20:>8.4f}   {ask20:>8.4f}   {imb20:>+8.2f}%   ', end='')
print('🟢' if imb20 > 10 else ('🔴' if imb20 < -10 else '⚪'))
print('=' * 55)

# 解读
if imb5 > 10:
    print('→ 短时买方强势，可能小拉升')
elif imb5 < -10:
    print('→ 短时卖方强势，可能小回调')
else:
    print('→ 前5档均衡，看更深度')

if imb20 > 15:
    print('→ 大资金在吸筹！深度买盘堆积')
elif imb20 < -15:
    print('→ 大资金在出货！深度卖盘堆积')
else:
    print('→ 整体均衡，没有明确的单边信号')

  BTC/USD | 10:11:50
  中间价: $62,401.54
  档位             买方深度       卖方深度       不平衡度     信号
  --------------------------------------------
  前5档      0.1256     0.1495      -8.69%   ⚪
  前10档     0.5821     0.6149      -2.74%   ⚪
  前20档     1.0493     1.3557     -12.74%   🔴
→ 前5档均衡，看更深度
→ 整体均衡，没有明确的单边信号
